In [6]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans
from MatrixFactorization import MatrixFactorization as mf
from CollaborativeFiltering import CollaborativeFiltering as cb

In [7]:
movie_ratings = pd.read_csv('./dataset/ratings.csv')
movie_ratings_small = pd.read_csv('./dataset/ratings_small.csv')
movies_metadata = pd.read_csv('./dataset/movies_metadata.csv')

/var/folders/s7/_6xph2kx7zj1t3c3tmb0gk5c0000gn/T/ipykernel_2194/2452708455.py:3: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  movies_metadata = pd.read_csv('./dataset/movies_metadata.csv')


In [8]:
movie_ratings_small.head()

,userId,movieId,rating,timestamp
0,1,31,2.5,1260759144
1,1,1029,3.0,1260759179
2,1,1061,3.0,1260759182
3,1,1129,2.0,1260759185
4,1,1172,4.0,1260759205


In [9]:
movie_ratings.shape, movie_ratings_small.shape

((26024289, 4), (100004, 4))

In [10]:
movies_metadata = movies_metadata.drop_duplicates(subset='id', keep='first')

In [11]:
movies_metadata = movies_metadata[movies_metadata['id'].str.isdigit()]
movies_metadata['id'] = movies_metadata['id'].astype('int64')

In [ ]:
# 두 데이터프레임 병합 (movieId 기준)
movies_total= movies_metadata.merge(movie_ratings,
                                        left_on='id',
                                        right_on='movieId',
                                        how='left')

# 필요 없어진 movieId 열 제거
movies_total = movies_total.drop(columns=['movieId'])


In [ ]:
movies_total.shape

In [ ]:
movies_total.dropna(subset='userId', inplace=True)

In [ ]:
movies_total['userId'] = movies_total['userId'].astype('int64')

In [ ]:
a =movies_total[['id','original_title','userId', 'rating']]
a.head()

In [ ]:
user_matrix = pd.pivot_table(a, index="userId", columns='original_title',values='rating',aggfunc="mean", fill_value=0)
user_matrix

In [ ]:
#kmeans를 사용하여 사용자-아이템 메트릭스를 클러스터링을 먼저 시행한다.
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans

# 예제 데이터 (사용자-아이템 행렬)
pca = PCA(n_components=10)
X_reduced = pca.fit_transform(user_matrix)

wcss = []
for k in range(1, 11):  # k를 1부터 10까지 테스트
    kmeans = MiniBatchKMeans(n_clusters=k, random_state=42, batch_size=500)
    kmeans.fit(X_reduced)
    wcss.append(kmeans.inertia_)  # inertia_ = WCSS 값

# 그래프 그리기
plt.plot(range(1, 11), wcss, marker='o')
plt.xlabel('클러스터 개수 k')
plt.ylabel('WCSS')
plt.title('Elbow Method for Optimal k')
plt.show()

In [ ]:
from sklearn.metrics import silhouette_score

silhouette_scores = []
pca = PCA(n_components=50)
X_reduced = pca.fit_transform(user_matrix)
silhouette_scores = []
# Silhouette Score 계산을 위한 샘플링 비율 설정 (예: 10% 샘플링)
sample_ratio =1
X_sampled = X_reduced[np.random.choice(X_reduced.shape[0], int(X_reduced.shape[0] * sample_ratio), replace=False)]
for k in range(2, 11):  # k=1은 의미 없으므로 2부터 시작
    kmeans = MiniBatchKMeans(n_clusters=k, random_state=42, batch_size=200, n_init=1)
    cluster_labels = kmeans.fit_predict(X_sampled)
    score = silhouette_score(X_sampled, cluster_labels)
    silhouette_scores.append(score)

# 그래프 그리기
plt.plot(range(2, 11), silhouette_scores, marker='o')
plt.xlabel('클러스터 개수 k')
plt.ylabel('Silhouette Score')

In [ ]:
mf = mf(user_matrix)
k_values = [100, 200, 300, 400,500]  # 실험할 k 값 리스트
for k in k_values:
    actual_items, predicted_items = mf.predict(k, top_k=10)
    precision = mf.precision_at_k(actual_items, predicted_items)
    print(f"k={k}, Precision: {precision:.4f}")


In [ ]:
k_values = [100, 200, 300, 400,500]  # 실험할 k 값 리스트
for k in k_values:
    actual_items, predicted_items = mf.predict(k, top_k=10)
    recall = mf.recall_at_k(actual_items, predicted_items)
    print(f"k={k}, Recall: {recall:.4f}")

In [ ]:
# 특정 사용자에 대해 유사도 계산
k = 2 # 위의결과로 설정됨
kmeans = MiniBatchKMeans(n_clusters=k, random_state=42, batch_size=500)
user_clusters = kmeans.fit_predict(user_matrix)
unique_clusters = np.unique(user_clusters)

user_id = 0  # 예: 첫 번째 사용자
cluster= user_clusters[0]

top_k = 10

cluster_indices = np.where(user_clusters == cluster)[0]  # 해당 클러스터 사용자 인덱스
cluster_matrix = user_matrix.iloc[cluster_indices]
optimal_k = 100
cb_= cb(cluster_matrix, optimal_k)
similarities = cb_.calculate_pearson_similarity(user_id)

In [ ]:
#similarities
sorted_similarities = sorted(similarities.items(), key=lambda x: x[1], reverse=True)
print(sorted_similarities)

In [ ]:
item_id = "장화, 홍련"
user_id = 1
predicted_rating = cb_.user_based_predict(user_id, item_id, similarities)
print(f"사용자 {user_id}가 '{item_id}'에 부여할 예측 평점: {predicted_rating}")